In [1]:
from openai import OpenAI
from decouple import config
import os
import pandas as pd
from retrying import retry

from tqdm.auto import tqdm
from tqdm.contrib.concurrent import thread_map
tqdm.pandas()


In [2]:
class OpenRouterWrapper:

  def __init__(self, model_name, max_tokens, temperature):
      self.model_name = model_name
      self.client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=config("OPENROUTER_API_KEY"),
      )
      self.max_tokens = max_tokens
      self.temperature = temperature

  @retry(wait_exponential_multiplier=1000, wait_exponential_max=10000)
  def get_completion(self, prompt):

    completion = self.client.chat.completions.create(
      model=self.model_name,
      messages=[
        {
          "role": "user",
          "content": prompt
        }
      ],
      max_tokens=self.max_tokens,
      temperature=self.temperature,
    )

    return completion.choices[0].message.content.strip()

  def get_parallel_completions(self, prompts, max_workers):
      completions = thread_map(self.get_completion, prompts, max_workers=max_workers)
      return [c for c in completions]
  

def collect_responses(df, eval_model):
    """
    Collect responses from the OpenRouter API for a given DataFrame of prompts.
    """
    
    df["eval_text"] = eval_model.get_parallel_completions(df["eval_prompt"], max_workers=8)
    df["eval_model"] = eval_model.model_name

    return df


def run_full_eval(dict, eval_model):
    """
    Run the full evaluation for all prompts in the dictionary.
    """
    
    # make target directory if it doesn't exist
    if not os.path.exists(f"./responses/{eval_model.model_name.split('/')[1]}"):
        os.makedirs(f"./responses/{eval_model.model_name.split('/')[1]}")

    print(f"Running full evaluation for {eval_model.model_name}...")

    # iterate over the dictionary and collect responses
    for key, df in sorted(dict.items()):
        print(f"Collecting responses for {key}...")
        results_df = collect_responses(df, eval_model)
        results_df.to_csv(f"./responses/{eval_model.model_name.split('/')[1]}/{key}.csv", index=False)

In [3]:
# load the eval prompts

eval_dict = {}

for file in os.listdir("./eval_prompts"):
  if file.endswith(".csv"):
    eval_dict[file.split("_")[2].strip(".csv")] = pd.read_csv(f"./eval_prompts/{file}")

In [4]:
# set up the eval model

eval_model = OpenRouterWrapper(
    model_name="openai/chatgpt-4o-latest",
    max_tokens=128,
    temperature=0.0
)

In [10]:
run_full_eval(eval_dict, eval_model)

Running full evaluation for openai/chatgpt-4o-latest...


  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]